In [2]:
from pathlib import Path

# Definimos la carpeta donde debería estar
folder = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025")

# Listamos qué archivos hay realmente en esa carpeta
print("Archivos encontrados en la carpeta:")
for archivo in folder.iterdir():
    print(archivo.name)

Archivos encontrados en la carpeta:
amazon_electronics_sample.csv


In [6]:
import pandas as pd
from pathlib import Path

# 1. Definimos la ruta completa del archivo
folder = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025")
ruta = folder / "amazon_electronics_sample.csv"

# 2. Cargar dataset
df = pd.read_csv(ruta)

# 3. Ver información inicial
print("Shape inicial:", df.shape)
print(df.head())
print(df.info())

# 4. Normalizar nombres de columnas
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# 5. Eliminar filas duplicadas
df = df.drop_duplicates()

# 6. Limpiar espacios y convertir valores nulos
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})

# 7. Eliminar columnas totalmente vacías
df = df.dropna(axis=1, how="all")

# 8. Rellenar o eliminar nulos (aquí podrías usar fillna() si prefieres)
df = df.dropna()

# 9. Intentar convertir columnas numéricas
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = pd.to_numeric(df[col], errors="ignore")

# 10. Resetear índice
df = df.reset_index(drop=True)

# 11. Resultado final y guardado
print("Shape final:", df.shape)
salida = folder / "amazon_electronics_sample_limpio.csv"
df.to_csv(salida, index=False)
print(f"Archivo limpio guardado en: {salida}")

Shape inicial: (300, 474)
           id                                              title     brand  \
0  B0BM8G3W11  XIAWAO USB to USB C Cable, (6-Pack, 4×6ft+2×9f...    XIAWAO   
1  B0CQLZWYF9  Pocket 3 Suction Cup Mount, Car Windshield Win...     BRDRC   
2  B0FKYKDLM7  Comfort Scroll Ring: True Touch Wireless Remot...    JLZNLC   
3  B01HCP9K10  67XL Printer Ink Compatible for HP Ink 67 Repl...  TOKYOINK   
4  B0BPNJVYS4  USB Multi Plug Outlet Extender - YISHU Surge P...     YISHU   

   price price_text  rating  review_count  \
0   9.99      $9.99     4.5        6737.0   
1  16.99     $16.99     4.3         383.0   
2  12.99     $12.99     5.0           7.0   
3  36.99     $36.99     4.2        2560.0   
4  10.99     $10.99     4.7        4780.0   

                                        availability  \
0                                           In Stock   
1                                           In Stock   
2                 Only 8 left in stock - order soon.   
3         

C:\Users\juanb\AppData\Local\Temp\ipykernel_19104\265598161.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [7]:
import pandas as pd
from pathlib import Path

# 1. Cargar
folder = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025")
ruta = folder / "amazon_electronics_sample.csv"
df = pd.read_csv(ruta)

# 2. Normalizar nombres
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# 3. Eliminar duplicados
df = df.drop_duplicates()

# 4. Limpieza de texto (corrigiendo la advertencia de pandas 3.0)
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].astype(str).str.strip().replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})

# 5. ELIMINAR columnas innecesarias antes de limpiar filas
# Muchas columnas son técnicas o tienen demasiados nulos. 
# Si no las vas a usar, elimínalas primero:
# df = df.drop(columns=['rank_2_category', 'rank_3_category', ...])

# 6. LIMPIEZA INTELIGENTE:
# Borra filas SÓLO si faltan datos críticos (id, title, price)
df = df.dropna(subset=['id', 'title', 'price'])

# 7. Rellenar nulos en lugar de borrar filas
# Para columnas de rating o reviews, es mejor poner 0 que borrar la fila
df['rating'] = df['rating'].fillna(0)
df['review_count'] = df['review_count'].fillna(0)

# 8. Guardar
salida = folder / "amazon_electronics_limpio.csv"
df.to_csv(salida, index=False)
print(f"Dataset guardado con {df.shape[0]} filas y {df.shape[1]} columnas.")

Dataset guardado con 291 filas y 474 columnas.


In [8]:
pip install sqlalchemy pandas

Note: you may need to restart the kernel to use updated packages.


In [9]:
from sqlalchemy import create_engine, text

# Creamos una base de datos llamada 'proyecto_gaming.db' en tu carpeta
# La librería crea el archivo automáticamente al conectar
engine = create_engine('sqlite:///proyecto_gaming.db')

# Probamos la conexión
try:
    with engine.connect() as connection:
        print("¡Conexión establecida con éxito!")
        # Ejecutamos una pequeña consulta de prueba
        result = connection.execute(text("SELECT 1"))
        print("Resultado de la prueba:", result.fetchone())
except Exception as e:
    print(f"Error al conectar: {e}")

¡Conexión establecida con éxito!
Resultado de la prueba: (1,)


In [1]:
import pandas as pd
from pathlib import Path

folder = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025")
ruta = folder / "amazon_electronics_sample.csv"
df = pd.read_csv(ruta)

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.drop_duplicates()

for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].astype(str).str.strip().replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})

cols_relevantes = [
    "id", "title", "brand", "price", "price_text", "rating", "review_count",
    "availability", "category_name", "category_id", "url", "description",
    "features_text", "images", "ranks"
]

cols_relevantes = [c for c in cols_relevantes if c in df.columns]
df_filtrado = df[cols_relevantes].copy()

df_filtrado = df_filtrado.dropna(axis=1, how="all")

salida = folder / "amazon_electronics_filtrado.csv"
df_filtrado.to_csv(salida, index=False)

print("Shape original:", df.shape)
print("Shape filtrado:", df_filtrado.shape)
print("Archivo guardado en:", salida)

Shape original: (300, 474)
Shape filtrado: (300, 15)
Archivo guardado en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025\amazon_electronics_filtrado.csv


In [2]:
import pandas as pd
import ast
from pathlib import Path

folder = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025")
ruta = folder / "amazon_electronics_sample.csv"

df = pd.read_csv(ruta)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

def safe_eval(x):
    if pd.isna(x):
        return []
    if isinstance(x, (list, dict)):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

subcategorias = []

for _, row in df.iterrows():
    product_id = row.get("id")
    title = row.get("title")
    category_name = row.get("category_name")

    if pd.notna(row.get("rank_1_category")):
        subcategorias.append({
            "id": product_id,
            "title": title,
            "category_name": category_name,
            "subcategory": row.get("rank_1_category"),
            "rank_position": row.get("rank_1_position"),
            "source": "rank_1_category"
        })

    if pd.notna(row.get("rank_2_category")):
        subcategorias.append({
            "id": product_id,
            "title": title,
            "category_name": category_name,
            "subcategory": row.get("rank_2_category"),
            "rank_position": row.get("rank_2_position"),
            "source": "rank_2_category"
        })

    if pd.notna(row.get("rank_3_category")):
        subcategorias.append({
            "id": product_id,
            "title": title,
            "category_name": category_name,
            "subcategory": row.get("rank_3_category"),
            "rank_position": row.get("rank_3_position"),
            "source": "rank_3_category"
        })

    ranks = safe_eval(row.get("ranks"))
    if isinstance(ranks, list):
        for item in ranks:
            if isinstance(item, dict):
                subcategorias.append({
                    "id": product_id,
                    "title": title,
                    "category_name": category_name,
                    "subcategory": item.get("category"),
                    "rank_position": item.get("rank"),
                    "source": "ranks"
                })

df_subcategorias = pd.DataFrame(subcategorias)

df_subcategorias = df_subcategorias.dropna(subset=["subcategory"])
df_subcategorias["subcategory"] = df_subcategorias["subcategory"].astype(str).str.strip()
df_subcategorias = df_subcategorias[df_subcategorias["subcategory"] != ""]
df_subcategorias = df_subcategorias.drop_duplicates().reset_index(drop=True)

salida = folder / "amazon_subcategorias_extraidas.csv"
df_subcategorias.to_csv(salida, index=False)

print("Shape subcategorías:", df_subcategorias.shape)
print(df_subcategorias.head())
print(f"Archivo guardado en: {salida}")

Shape subcategorías: (832, 6)
           id                                              title  \
0  B0BM8G3W11  XIAWAO USB to USB C Cable, (6-Pack, 4×6ft+2×9f...   
1  B0BM8G3W11  XIAWAO USB to USB C Cable, (6-Pack, 4×6ft+2×9f...   
2  B0CQLZWYF9  Pocket 3 Suction Cup Mount, Car Windshield Win...   
3  B0CQLZWYF9  Pocket 3 Suction Cup Mount, Car Windshield Win...   
4  B0FKYKDLM7  Comfort Scroll Ring: True Touch Wireless Remot...   

                           category_name  \
0                        cables_12954861   
1                        cables_12954861   
2  action_camera_accessories_75364150011   
3  action_camera_accessories_75364150011   
4               remote_controls_14015071   

                                         subcategory  rank_position  \
0                                         USB Cables           39.0   
1                                         USB Cables           39.0   
2  Action Camera Accessories        Special featu...           14.0   
3  Action Ca

In [3]:
import pandas as pd
from pathlib import Path

# 1. Ajusta esta ruta si fuera necesario, pero esta es la que vimos que funcionaba en tu equipo
folder = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025")
ruta = folder / "amazon_electronics_sample.csv"

# 2. Cargar el dataset
df = pd.read_csv(ruta)

# 3. Definir palabras clave específicas para gaming
keywords = ['monitor', 'pc', 'controller', 'mouse', 'keyboard', 'headset', 'headphones']

# 4. Filtrar: busca esas palabras en título o categoría
# Convertimos a string por seguridad y luego a minúsculas
df['search_text'] = (df['title'].astype(str) + " " + df['category_name'].astype(str)).str.lower()

# Filtramos las filas que contienen al menos una de las palabras
df_gaming = df[df['search_text'].str.contains('|'.join(keywords), na=False)].copy()

# Eliminar la columna auxiliar
df_gaming = df_gaming.drop(columns=['search_text'])

# 5. Guardar el nuevo dataset filtrado
salida = folder / "amazon_electronics_gaming.csv"
df_gaming.to_csv(salida, index=False)

print(f"Dataset original: {len(df)} filas")
print(f"Dataset filtrado (Gaming): {len(df_gaming)} filas")
print(f"Archivo guardado en: {salida}")

Dataset original: 300 filas
Dataset filtrado (Gaming): 31 filas
Archivo guardado en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025\amazon_electronics_gaming.csv


C:\Users\juanb\AppData\Local\Temp\ipykernel_22172\2452640257.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['search_text'] = (df['title'].astype(str) + " " + df['category_name'].astype(str)).str.lower()


In [4]:
import pandas as pd
from pathlib import Path

# 1. Configuración de rutas
folder = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025")
ruta = folder / "amazon_electronics_sample.csv"

# 2. Cargar datos
df = pd.read_csv(ruta)

# 3. Filtrado inicial de gaming
keywords = ['monitor', 'pc', 'controller', 'mouse', 'keyboard', 'headset', 'headphones']
df['search_text'] = (df['title'].astype(str) + " " + df['category_name'].astype(str)).str.lower()
df_gaming = df[df['search_text'].str.contains('|'.join(keywords), na=False)].copy()

# 4. LIMPIEZA POST-FILTRADO (Duplicados y Nulos)
# Eliminamos duplicados basados en el 'id' (si un mismo producto aparece dos veces)
df_gaming = df_gaming.drop_duplicates(subset=['id'])

# Eliminamos filas donde datos críticos sean nulos (ej: no tenemos precio ni título)
df_gaming = df_gaming.dropna(subset=['title', 'price', 'id'])

# Opcional: rellenar nulos en columnas de rating/reviews con 0
if 'rating' in df_gaming.columns:
    df_gaming['rating'] = df_gaming['rating'].fillna(0)
if 'review_count' in df_gaming.columns:
    df_gaming['review_count'] = df_gaming['review_count'].fillna(0)

# Limpiar columna auxiliar
df_gaming = df_gaming.drop(columns=['search_text'])

# 5. Guardar resultado limpio
salida = folder / "amazon_electronics_gaming_limpio.csv"
df_gaming.to_csv(salida, index=False)

print(f"Dataset original: {len(df)} filas")
print(f"Dataset gaming limpio: {len(df_gaming)} filas")
print(f"Archivo guardado en: {salida}")

Dataset original: 300 filas
Dataset gaming limpio: 30 filas
Archivo guardado en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Amazon Best-Selling Electronics Dataset 2025\amazon_electronics_gaming_limpio.csv


C:\Users\juanb\AppData\Local\Temp\ipykernel_22172\930133607.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['search_text'] = (df['title'].astype(str) + " " + df['category_name'].astype(str)).str.lower()
